# SAA-CVaR, DR-CVaR, RS-CVaR, and RSDR-CVaR Evaluation

This notebook evaluates how four tail-risk portfolio optimizers behave on the ETF trading universe using the selected upstream regime source: `altdata_bnmeanrank_hmm`.

The purpose is not to re-estimate the regime model here. This copy intentionally restricts the regime input to `altdata_bnmeanrank_hmm`, then asks:

> Given the historical ETF returns and the already-estimated regime probabilities, how would each CVaR optimizer allocate the ETF portfolio month by month?

The four strategies compared are:

1. **SAA-CVaR**: uses the empirical historical ETF return distribution directly.
2. **DR-CVaR**: starts from the same empirical distribution, then protects against nearby plausible distributions inside a Wasserstein ball.
3. **RS-CVaR**: uses the regime model to reweight historical scenarios by the predicted next-regime probabilities.
4. **RSDR-CVaR**: combines both ideas: regime-weighted scenarios plus a Wasserstein robust penalty inside the regime-aware objective.

The optimizer always chooses a portfolio weight vector \(x\). The regime files do **not** become tradable assets. They only affect how historical ETF return scenarios are weighted before the optimizer chooses ETF allocations.


## 1. Data, Formation Windows, and Out-of-Sample Timing

### Traded assets

The traded universe is taken from `data/etf_returns.csv`. In this run, the ETF return columns are:

`SPY, XLB, XLE, XLF, XLI, XLK, XLP, XLU, XLV, XLY`

So the number of tradable assets is:

$$
I = 10.
$$

At every rebalance date, the optimizer chooses ETF weights:

$$
x =
\begin{bmatrix}
x_1 \\
x_2 \\
\vdots \\
x_I
\end{bmatrix},
\qquad
\sum_{j=1}^{I}x_j=1,
\qquad
0 \leq x_j \leq 1.
$$

The long-only constraint means the portfolio can allocate across ETFs but cannot short them.

### Return scenario

One historical monthly observation is one ETF return vector:

$$
R_n =
\begin{bmatrix}
R_{n,1} \\
R_{n,2} \\
\vdots \\
R_{n,I}
\end{bmatrix}
\in \mathbb{R}^{I}.
$$

Each entry is the return of one ETF in the same month. Therefore, one row of `etf_returns.csv` is one historical market scenario across all ETFs.

### CVaR formation window

The CVaR optimizer uses a rolling formation window of:

$$
N = 120 \text{ months}.
$$

That means each monthly decision is trained using the previous 120 monthly ETF return vectors:

$$
\mathcal{R}_t =
\{R_{t-120}, R_{t-119}, \ldots, R_{t-1}\}.
$$

Month \(t\) itself is not used when solving for \(x_t\). It is only used afterward to evaluate the realized out-of-sample return.

### Rebalance and prediction horizon

The rebalance step is:

$$
\text{REBALANCE\_STEP}=1.
$$

So the notebook re-optimizes once per month. The portfolio chosen at month \(t\) is held for that next monthly return observation:

$$
r^{\mathrm{portfolio}}_t = x_t^\top R_t.
$$

Because the ETF data and regime-state files are aligned by month, the practical out-of-sample evaluation begins after the first 120 aligned months. With this dataset, that produces roughly 108 out-of-sample monthly portfolio returns.

### Regime formation window

The regime labels and regime probabilities are not estimated inside this notebook. This copy loads only the selected upstream walk-forward regime file:

`outputs/states_altdata_bnmeanrank_hmm_walkforward.csv`

Each file supplies:

- a discrete regime label, `state`;
- state probability columns such as `p_state0`, `p_state1`, ..., depending on the regime source.

The notebook then uses the previous month's regime probability vector and the rolling historical state sequence to compute next-regime weights for RS-CVaR and RSDR-CVaR.


## 2. Optimizer Formulas and Intuition

### Portfolio return and portfolio loss

For a chosen ETF weight vector \(x\), the portfolio return under historical scenario \(R_n\) is:

$$
x^\top R_n.
$$

Because CVaR is written as a loss minimization problem, the corresponding portfolio loss is:

$$
\ell(x,R_n) = -x^\top R_n.
$$

If \(x^\top R_n\) is negative, the loss is positive. CVaR focuses on the bad right tail of this loss distribution.

The VaR threshold is denoted by \(v\). The CVaR tail-loss excess term is:

$$
[\ell(x,R_n)-v]_+
=
\max(\ell(x,R_n)-v,0).
$$

Only losses worse than \(v\) contribute to the CVaR tail average.

---

### SAA-CVaR: empirical tail-risk minimization

The empirical distribution places equal probability mass on every scenario in the 120-month formation window:

$$
\widehat{P}_N
=
\frac{1}{N}\sum_{n=1}^{N}\delta_{R_n}.
$$

The SAA-CVaR optimizer solves:

$$
\min_{x\in\mathcal{X},\,v\in\mathbb{R}}
\left\{
v
+
\frac{1}{N(1-\eta)}
\sum_{n=1}^{N}
[\ell(x,R_n)-v]_+
\right\}.
$$

**Intuition.** SAA-CVaR trusts the historical sample exactly. It asks: among the past 120 ETF return scenarios, which ETF weights minimize the average loss beyond the \(\eta=95\%\) VaR threshold?

**Limitation.** If the 120-month sample under-represents a crisis or sampling error, SAA-CVaR can be too confident in the observed history.

---

### DR-CVaR: protect against nearby plausible distributions

DR-CVaR keeps the same empirical center \(\widehat{P}_N\), but does not fully trust it. It considers all distributions \(P\) that are close to the empirical distribution under Wasserstein distance:

$$
\mathcal{U}^{w}(\theta)
=
\{P: W_m(P,\widehat{P}_N)\leq \theta\}.
$$

The high-level robust objective is:

$$
\min_{x,v}
\left\{
v+
\frac{1}{1-\eta}
\sup_{P\in\mathcal{U}^{w}(\theta)}
\mathbb{E}^{P}
[\ell(x,R)-v]_+
\right\}.
$$

In this notebook, the `l1` Wasserstein ground norm is used. Its dual norm is \(\|x\|_\infty\). Therefore the tractable implementation becomes empirical CVaR plus a robust penalty:

$$
\min_{x,v}
\left\{
v
+
\frac{1}{1-\eta}
\sum_{n=1}^{N}p_n[\ell(x,R_n)-v]_+
+
\frac{\theta}{1-\eta}\|x\|_\infty
\right\}.
$$

For plain DR-CVaR:

$$
p_n=\frac{1}{N},
\qquad
\theta = \gamma N^{-1/I}.
$$

**Intuition.** The Wasserstein term says: do not only minimize the observed historical tail loss; also pay a penalty if the portfolio is fragile to nearby changes in the return distribution. With the `l1` ground norm, concentrated portfolios are penalized through \(\|x\|_\infty\).

---

### RS-CVaR: regime-weighted empirical CVaR

RS-CVaR uses the regime model to change the scenario probabilities. The notebook first estimates a rolling transition matrix from the historical state sequence:

$$
A_t(i,j)
=
\Pr(S_{t}=j\mid S_{t-1}=i).
$$

The previous month's state probability vector is:

$$
\pi_{t-1}
=
\begin{bmatrix}
\Pr(S_{t-1}=1) & \cdots & \Pr(S_{t-1}=K)
\end{bmatrix}.
$$

The predicted next-regime weights are:

$$
w_t = \pi_{t-1}A_t.
$$

If historical scenario \(n\) belongs to regime \(k\), and there are \(N_k\) scenarios from regime \(k\) inside the 120-month window, RS-CVaR assigns:

$$
p_n
=
\frac{w_{t,k}}{N_k},
\qquad
\text{for } s_n=k.
$$

The RS-CVaR optimizer solves:

$$
\min_{x,v}
\left\{
v+
\frac{1}{1-\eta}
\sum_{n=1}^{N}
p_n[\ell(x,R_n)-v]_+
\right\}.
$$

**Intuition.** RS-CVaR still uses historical ETF scenarios, but it does not treat every past month equally. If the regime model says the next month is more likely to resemble a particular regime, the historical months belonging to that regime receive more probability mass.

---

### RSDR-CVaR: regime-weighted CVaR with Wasserstein robustness

RSDR-CVaR combines the RS and DR ideas. Each regime has its own empirical distribution:

$$
\widehat{P}_k
=
\frac{1}{N_k}
\sum_{n:s_n=k}
\delta_{R_n}.
$$

Each regime also has its own Wasserstein radius:

$$
\theta_k = \gamma N_k^{-1/I}.
$$

The high-level RSDR objective is:

$$
\min_{x,v}
\left\{
v+
\frac{1}{1-\eta}
\sum_{k=1}^{K}
w_{t,k}
\sup_{P_k\in \mathcal{U}^{w}_k(\theta_k)}
\mathbb{E}^{P_k}
[\ell(x,R)-v]_+
\right\}.
$$

The implemented `l1`-norm equivalent is:

$$
\min_{x,v}
\left\{
v
+
\frac{1}{1-\eta}
\sum_{n=1}^{N}
p_n[\ell(x,R_n)-v]_+
+
\frac{\theta_{\mathrm{RSDR},t}}{1-\eta}\|x\|_\infty
\right\},
$$

where:

$$
\theta_{\mathrm{RSDR},t}
=
\sum_{k=1}^{K}w_{t,k}\theta_k.
$$

**Intuition.** RS-CVaR asks which regime is likely next. DR-CVaR asks what if the empirical distribution is slightly wrong. RSDR-CVaR asks both questions together: which regime is likely next, and within those regime-specific return scenarios, what if the distribution is slightly worse than observed?


## 3. Wasserstein Radius and Paper Gamma Grid

The notebook uses the paper-style ambiguity-aversion grid:

$$
\gamma \in \{0.02,\ 0.04,\ 0.06,\ 0.08,\ 0.10\}.
$$

The actual Wasserstein radius is not set equal to \(\gamma\). It is scaled by the number of observations and number of assets:

$$
\theta = \gamma N^{-1/I}.
$$

For this notebook:

- \(N=120\) in the standard CVaR formation window;
- \(I=10\) traded ETFs when `SPY` is included;
- \(\eta=0.95\);
- `ROBUST_GROUND_NORM = "l1"`.

So increasing \(\gamma\) increases the size of the Wasserstein ambiguity set. A larger \(\gamma\) makes DR-CVaR and RSDR-CVaR more conservative because the optimizer must perform well against a wider set of plausible distributions.

The output folder is:

`outputs/cvar_evaluation_paper_gamma_grid_altdata_bnmeanrank_hmm/`


In [ ]:

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linprog, minimize

warnings.filterwarnings("ignore")

def find_project_dir(start=None):
    """Find the project folder without hardcoding a user-specific path."""
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    candidates = [start, *start.parents]

    for base in candidates:
        if (base / "data" / "etf_returns.csv").exists():
            return base
        if (base / "Data" / "etf_returns.csv").exists():
            return base

    for base in candidates[:3]:
        for match in base.glob("**/etf_returns.csv"):
            if match.parent.name.lower() == "data":
                return match.parent.parent

    raise FileNotFoundError(
        "Could not find data/etf_returns.csv. Open this notebook from the project root, "
        "or place the notebook inside the project folder."
    )

PROJECT_DIR = find_project_dir()
DATA_DIR = PROJECT_DIR / ("data" if (PROJECT_DIR / "data").exists() else "Data")
REGIME_DIR = PROJECT_DIR / "outputs"
SELECTED_REGIME_SOURCE = "altdata_bnmeanrank_hmm"
OUT_DIR = PROJECT_DIR / "outputs" / f"cvar_evaluation_paper_gamma_grid_{SELECTED_REGIME_SOURCE}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ETA = 0.95
TRAIN_WINDOW = 120
REBALANCE_STEP = 1
GAMMA_GRID = [0.02, 0.04, 0.06, 0.08, 0.10]
ROBUST_GROUND_NORM = "l1"   # "l1" is the default tractable Wasserstein setup used here.
MAX_WEIGHT = 1.0
INCLUDE_SPY_AS_TRADABLE = True
USE_WALKFORWARD_STATES = True

REGIME_FILE_SUFFIX = "walkforward" if USE_WALKFORWARD_STATES else "fullsample"
REGIME_FILES = {
    SELECTED_REGIME_SOURCE: REGIME_DIR / f"states_{SELECTED_REGIME_SOURCE}_{REGIME_FILE_SUFFIX}.csv",
}

print("Project directory:", PROJECT_DIR)
print("Output directory:", OUT_DIR)
print("Paper gamma grid:", GAMMA_GRID)
print("Regime files:")
for name, path in REGIME_FILES.items():
    print(f"  {name}: {path.name}  exists={path.exists()}")


In [ ]:
def load_etf_returns():
    path = DATA_DIR / "etf_returns.csv"
    returns = pd.read_csv(path)
    date_col = "month" if "month" in returns.columns else returns.columns[0]
    returns[date_col] = pd.to_datetime(returns[date_col]).dt.to_period("M").dt.to_timestamp()
    returns = returns.set_index(date_col).sort_index()
    returns = returns.apply(pd.to_numeric, errors="coerce").dropna(how="any")
    if not INCLUDE_SPY_AS_TRADABLE and "SPY" in returns.columns:
        returns = returns.drop(columns=["SPY"])
    return returns


def load_regime_file(path):
    df = pd.read_csv(path)

    if "date" in df.columns:
        date_col = "date"
    elif "sasdate" in df.columns:
        date_col = "sasdate"
    elif "month" in df.columns:
        date_col = "month"
    else:
        date_col = df.columns[0]

    df[date_col] = pd.to_datetime(df[date_col]).dt.to_period("M").dt.to_timestamp()
    df = df.set_index(date_col).sort_index()

    state_cols = [c for c in df.columns if c.startswith("p_state")]
    if not state_cols:
        raise ValueError(f"No p_state columns found in {path}")

    def parse_state_col(col):
        return int(col.replace("p_state_", "").replace("p_state", ""))

    probs = df[state_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)
    probs.columns = [parse_state_col(c) for c in probs.columns]
    probs = probs.reindex(sorted(probs.columns), axis=1)
    probs = probs.div(probs.sum(axis=1).replace(0.0, np.nan), axis=0)
    probs = probs.fillna(1.0 / probs.shape[1])

    state = pd.to_numeric(df["state"], errors="coerce").astype("Int64")
    out = pd.DataFrame(index=df.index)
    out["state"] = state
    if "crisis" in df.columns:
        out["crisis"] = pd.to_numeric(df["crisis"], errors="coerce")
    if "p_crisis" in df.columns:
        out["p_crisis"] = pd.to_numeric(df["p_crisis"], errors="coerce")
    return out, probs


returns = load_etf_returns()
asset_cols = returns.columns.tolist()

regime_inputs = {}
for name, path in REGIME_FILES.items():
    if path.exists():
        labels, probs = load_regime_file(path)
        regime_inputs[name] = {"labels": labels, "probs": probs}

print("ETF return matrix:", returns.shape, returns.index.min().date(), "to", returns.index.max().date())
print("Assets:", asset_cols)
print()
for name, data in regime_inputs.items():
    labels, probs = data["labels"], data["probs"]
    print(f"{name}: labels={labels.shape}, probs={probs.shape}, range={labels.index.min().date()} to {labels.index.max().date()}")

In [ ]:
def theta_from_gamma(gamma, n_obs, n_assets):
    # Paper-style radius rule: theta = gamma * N^(-1/I).
    return float(gamma * max(n_obs, 1) ** (-1.0 / max(n_assets, 1)))


def normalize_probs(p):
    p = np.asarray(p, dtype=float)
    p = np.clip(p, 0.0, None)
    s = p.sum()
    if s <= 0:
        return np.ones_like(p) / len(p)
    return p / s


def solve_weighted_cvar_lp_l1(R, obs_probs, eta, theta_eff=0.0, max_weight=1.0):
    # Solve weighted CVaR with optional l1-Wasserstein robust penalty.
    # For a long-only portfolio, the l1 ground norm yields the dual ||x||_infinity.
    # We introduce z with x_j <= z and add theta_eff/(1-eta) * z to the objective.
    R = np.asarray(R, dtype=float)
    n, i_assets = R.shape
    obs_probs = normalize_probs(obs_probs)

    ix_x0 = 0
    ix_v = i_assets
    ix_u0 = i_assets + 1
    ix_z = i_assets + 1 + n
    n_var = i_assets + 1 + n + 1

    c = np.zeros(n_var)
    c[ix_v] = 1.0
    c[ix_u0:ix_u0 + n] = obs_probs / (1.0 - eta)
    c[ix_z] = theta_eff / (1.0 - eta)

    A_ub, b_ub = [], []

    # u_n >= -x'R_n - v
    for row_i in range(n):
        row = np.zeros(n_var)
        row[ix_x0:ix_x0 + i_assets] = -R[row_i]
        row[ix_v] = -1.0
        row[ix_u0 + row_i] = -1.0
        A_ub.append(row)
        b_ub.append(0.0)

    # z >= x_j
    for j in range(i_assets):
        row = np.zeros(n_var)
        row[ix_x0 + j] = 1.0
        row[ix_z] = -1.0
        A_ub.append(row)
        b_ub.append(0.0)

    A_eq = np.zeros((1, n_var))
    A_eq[0, ix_x0:ix_x0 + i_assets] = 1.0
    b_eq = np.array([1.0])

    bounds = [(0.0, max_weight)] * i_assets + [(None, None)] + [(0.0, None)] * n + [(0.0, max_weight)]

    res = linprog(
        c,
        A_ub=np.asarray(A_ub),
        b_ub=np.asarray(b_ub),
        A_eq=A_eq,
        b_eq=b_eq,
        bounds=bounds,
        method="highs",
    )

    if not res.success:
        # Robust fallback: equal weight, rather than crashing the whole backtest.
        x = np.ones(i_assets) / i_assets
        loss = -R @ x
        v = float(np.quantile(loss, eta))
        u = np.maximum(loss - v, 0.0)
        obj = float(v + np.sum(obs_probs * u) / (1.0 - eta) + theta_eff * np.max(x) / (1.0 - eta))
        return {
            "success": False,
            "message": res.message,
            "x": x,
            "v": v,
            "objective": obj,
            "z": float(np.max(x)),
        }

    x = res.x[ix_x0:ix_x0 + i_assets]
    return {
        "success": True,
        "message": res.message,
        "x": x,
        "v": float(res.x[ix_v]),
        "objective": float(res.fun),
        "z": float(res.x[ix_z]),
    }


def solve_weighted_cvar_l2_penalty(R, obs_probs, eta, theta_eff=0.0, max_weight=1.0):
    # Optional l2 version. Slower but useful for sensitivity checks.
    R = np.asarray(R, dtype=float)
    n, i_assets = R.shape
    obs_probs = normalize_probs(obs_probs)

    def objective(y):
        x = y[:i_assets]
        v = y[i_assets]
        losses = -R @ x
        tail = np.maximum(losses - v, 0.0)
        return v + np.sum(obs_probs * tail) / (1.0 - eta) + theta_eff * np.linalg.norm(x, 2) / (1.0 - eta)

    x0 = np.ones(i_assets) / i_assets
    loss0 = -R @ x0
    y0 = np.r_[x0, np.quantile(loss0, eta)]
    constraints = [{"type": "eq", "fun": lambda y: np.sum(y[:i_assets]) - 1.0}]
    bounds = [(0.0, max_weight)] * i_assets + [(None, None)]
    res = minimize(objective, y0, method="SLSQP", bounds=bounds, constraints=constraints, options={"maxiter": 1000, "ftol": 1e-10})
    x = res.x[:i_assets] if res.success else x0
    v = float(res.x[i_assets]) if res.success else float(np.quantile(loss0, eta))
    return {
        "success": bool(res.success),
        "message": str(res.message),
        "x": np.clip(x, 0.0, max_weight) / np.clip(x, 0.0, max_weight).sum(),
        "v": v,
        "objective": float(objective(np.r_[x, v])),
        "z": float(np.linalg.norm(x, 2)),
    }


def solve_cvar(R, obs_probs, eta, theta_eff=0.0, ground_norm="l1", max_weight=1.0):
    if ground_norm.lower() == "l2":
        return solve_weighted_cvar_l2_penalty(R, obs_probs, eta, theta_eff, max_weight)
    return solve_weighted_cvar_lp_l1(R, obs_probs, eta, theta_eff, max_weight)

In [ ]:
def estimate_transition_matrix(states, state_values, smoothing=1e-3):
    states = pd.Series(states).dropna().astype(int)
    k = len(state_values)
    idx = {s: i for i, s in enumerate(state_values)}
    mat = np.full((k, k), smoothing, dtype=float)
    vals = states.values
    for a, b in zip(vals[:-1], vals[1:]):
        if a in idx and b in idx:
            mat[idx[a], idx[b]] += 1.0
    mat = mat / mat.sum(axis=1, keepdims=True)
    return pd.DataFrame(mat, index=state_values, columns=state_values)


def next_regime_weights(prev_month, probs, train_states, state_values):
    pi = probs.loc[prev_month].reindex(state_values).fillna(0.0)
    pi = pi / pi.sum() if pi.sum() > 0 else pd.Series(1.0 / len(state_values), index=state_values)
    A = estimate_transition_matrix(train_states, state_values)
    w = pi.values @ A.loc[pi.index, state_values].values
    out = pd.Series(w, index=state_values).clip(lower=0.0)
    return out / out.sum(), A


def observation_probs_from_regime_weights(train_states, regime_weights):
    # Convert regime-level weights into observation-level probabilities.
    # If regime k receives weight w_k and has N_k historical scenarios in the window,
    # each scenario in that regime receives mass w_k / N_k.
    train_states = pd.Series(train_states).astype(int)
    obs_probs = pd.Series(0.0, index=train_states.index, dtype=float)
    allocated = 0.0
    missing_weight = 0.0

    for k, wk in regime_weights.items():
        mask = train_states == int(k)
        count = int(mask.sum())
        if count > 0:
            obs_probs.loc[mask] = float(wk) / count
            allocated += float(wk)
        else:
            missing_weight += float(wk)

    if obs_probs.sum() <= 0:
        obs_probs[:] = 1.0 / len(obs_probs)
    else:
        # If the predicted next regime has no historical observations in the
        # training window, redistribute that small weight uniformly.
        if missing_weight > 0:
            obs_probs += missing_weight / len(obs_probs)
        obs_probs = obs_probs / obs_probs.sum()

    return obs_probs.values


def regime_weighted_theta(gamma, train_states, regime_weights, n_assets):
    # Aggregate regime-specific theta_k = gamma * N_k^(-1/I).
    train_states = pd.Series(train_states).astype(int)
    theta = 0.0
    for k, wk in regime_weights.items():
        nk = int((train_states == int(k)).sum())
        if nk > 0 and wk > 0:
            theta += float(wk) * theta_from_gamma(gamma, nk, n_assets)
    return float(theta)


def effective_n_assets(weights):
    weights = np.asarray(weights, dtype=float)
    return 1.0 / np.sum(np.square(weights)) if np.sum(np.square(weights)) > 0 else np.nan

In [ ]:

def gamma_label(gamma):
    return str(gamma).replace(".", "p")


all_returns = []
all_weights = []
all_diagnostics = []

R_all = returns.copy()
n_assets = len(asset_cols)

for gamma in GAMMA_GRID:
    print(f"\n=== Running paper gamma = {gamma:.3f} ===")

    for regime_name, data in regime_inputs.items():
        labels = data["labels"]
        probs = data["probs"]

        common = R_all.index.intersection(labels.index).intersection(probs.index)
        R = R_all.loc[common]
        labels = labels.loc[common]
        probs = probs.loc[common]
        state_values = sorted(probs.columns.tolist())

        if len(R) <= TRAIN_WINDOW:
            print(f"Skipping {regime_name}: only {len(R)} aligned months, TRAIN_WINDOW={TRAIN_WINDOW}")
            continue

        print(f"Running {regime_name}: {len(R)} aligned months from {R.index.min().date()} to {R.index.max().date()}")

        for i in range(TRAIN_WINDOW, len(R), REBALANCE_STEP):
            rebalance_month = R.index[i]
            prev_month = R.index[i - 1]
            train_R = R.iloc[i - TRAIN_WINDOW:i]
            test_r = R.iloc[i]
            train_states = labels["state"].iloc[i - TRAIN_WINDOW:i].astype(int)

            n_train = len(train_R)
            equal_obs_probs = np.ones(n_train) / n_train
            theta_dr = theta_from_gamma(gamma, n_train, n_assets)

            regime_w, A_rolling = next_regime_weights(prev_month, probs, train_states, state_values)
            rs_obs_probs = observation_probs_from_regime_weights(train_states, regime_w)
            theta_rsdr = regime_weighted_theta(gamma, train_states, regime_w, n_assets)

            strategy_specs = {
                "SAA-CVaR": {
                    "obs_probs": equal_obs_probs,
                    "theta": 0.0,
                },
                "DR-CVaR": {
                    "obs_probs": equal_obs_probs,
                    "theta": theta_dr,
                },
                "RS-CVaR": {
                    "obs_probs": rs_obs_probs,
                    "theta": 0.0,
                },
                "RSDR-CVaR": {
                    "obs_probs": rs_obs_probs,
                    "theta": theta_rsdr,
                },
            }

            for strategy, spec in strategy_specs.items():
                sol = solve_cvar(
                    train_R.values,
                    spec["obs_probs"],
                    ETA,
                    theta_eff=spec["theta"],
                    ground_norm=ROBUST_GROUND_NORM,
                    max_weight=MAX_WEIGHT,
                )

                w = sol["x"]
                realized_return = float(np.dot(w, test_r.values))
                predicted_state = int(labels["state"].loc[rebalance_month])
                predicted_state_prob = float(probs.loc[prev_month].max())

                all_returns.append({
                    "gamma": gamma,
                    "regime_source": regime_name,
                    "month": rebalance_month,
                    "strategy": strategy,
                    "return": realized_return,
                    "predicted_state": predicted_state,
                })

                weight_row = {
                    "gamma": gamma,
                    "regime_source": regime_name,
                    "rebalance_month": rebalance_month,
                    "strategy": strategy,
                }
                weight_row.update({asset: float(weight) for asset, weight in zip(asset_cols, w)})
                all_weights.append(weight_row)

                diag = {
                    "gamma": gamma,
                    "regime_source": regime_name,
                    "rebalance_month": rebalance_month,
                    "strategy": strategy,
                    "success": sol["success"],
                    "objective": sol["objective"],
                    "VaR_v": sol["v"],
                    "theta_used": spec["theta"],
                    "robust_norm_value": sol["z"],
                    "robust_penalty_component": spec["theta"] * sol["z"] / (1.0 - ETA),
                    "max_weight": float(np.max(w)),
                    "effective_n_assets": float(effective_n_assets(w)),
                    "predicted_state": predicted_state,
                    "predicted_state_prob": predicted_state_prob,
                    "regime_weight_max": float(regime_w.max()),
                    "regime_weight_argmax": int(regime_w.idxmax()),
                    "message": sol["message"],
                }
                all_diagnostics.append(diag)

strategy_returns = pd.DataFrame(all_returns)
strategy_weights = pd.DataFrame(all_weights)
diagnostics = pd.DataFrame(all_diagnostics)

print("Strategy return rows:", strategy_returns.shape)
print("Strategy weight rows:", strategy_weights.shape)
print("Diagnostics rows:", diagnostics.shape)
display(strategy_returns.head())


## 4. Performance Metrics Computed by the Next Code Cell

The next code cell takes the realized out-of-sample monthly strategy returns and converts them into the comparison table saved as:

`performance_metrics.csv`

Let the realized monthly return of a strategy be \(r_t\), for \(t=1,\ldots,T\). The cumulative wealth path is:

$$
W_t = \prod_{\tau=1}^{t}(1+r_\tau).
$$

### Number of out-of-sample months

$$
T = \text{number of realized monthly strategy returns}.
$$

This is reported as `n_months`.

### Annualized return

The notebook annualizes monthly compounded wealth using:

$$
R_{\mathrm{ann}}
=
W_T^{12/T}-1.
$$

**Intuition.** This asks: if the realized monthly compounding rate persisted for a full year, what yearly return would it imply?

### Annualized volatility

Monthly volatility is annualized as:

$$
\sigma_{\mathrm{ann}}
=
\sqrt{12}\cdot \operatorname{Std}(r_t).
$$

**Intuition.** This measures how unstable the realized monthly returns are, expressed on an annual scale.

### Sharpe ratio

The notebook uses a zero risk-free rate approximation:

$$
\mathrm{Sharpe}
=
\frac{R_{\mathrm{ann}}}{\sigma_{\mathrm{ann}}}.
$$

**Intuition.** A higher Sharpe means more annualized return per unit of annualized volatility.

### Maximum drawdown

The running peak wealth is:

$$
\overline{W}_t = \max_{\tau\leq t} W_\tau.
$$

The drawdown at time \(t\) is:

$$
D_t = \frac{W_t}{\overline{W}_t}-1.
$$

The maximum drawdown is:

$$
\mathrm{MDD}=\min_t D_t.
$$

**Intuition.** This is the worst peak-to-trough loss experienced by the strategy.

### VaR and CVaR of realized strategy losses

The realized strategy loss is:

$$
L_t = -r_t.
$$

The 95% historical VaR is:

$$
\mathrm{VaR}_{0.95}
=
Q_{0.95}(L_t).
$$

The realized 95% CVaR is:

$$
\mathrm{CVaR}_{0.95}
=
\mathbb{E}\left[L_t \mid L_t \geq \mathrm{VaR}_{0.95}\right].
$$

**Intuition.** VaR marks the beginning of the bad tail. CVaR measures the average loss inside that bad tail.

### Final wealth

Final wealth is:

$$
W_T = \prod_{t=1}^{T}(1+r_t).
$$

It is reported as `final_wealth`, where a value of 2.0 means one dollar became two dollars.

### Turnover

Let \(x_t\) be the ETF weight vector chosen at rebalance month \(t\). The one-period turnover is:

$$
\mathrm{Turnover}_t
=
\sum_{j=1}^{I}
\left|x_{t,j}-x_{t-1,j}\right|.
$$

The notebook reports:

$$
\mathrm{Average\ Turnover}
=
\frac{1}{T-1}\sum_{t=2}^{T}\mathrm{Turnover}_t,
$$

and:

$$
\mathrm{Maximum\ Turnover}
=
\max_t \mathrm{Turnover}_t.
$$

**Intuition.** Turnover measures how aggressively the optimizer changes ETF allocations from one rebalance month to the next. Higher turnover can imply higher trading cost sensitivity, even though transaction costs are not explicitly deducted in this notebook.


In [ ]:

def max_drawdown(ret):
    ret = pd.Series(ret).dropna()
    if ret.empty:
        return np.nan
    wealth = (1.0 + ret).cumprod()
    peak = wealth.cummax()
    return float((wealth / peak - 1.0).min())


def empirical_var_cvar(ret, eta=0.95):
    ret = pd.Series(ret).dropna()
    if ret.empty:
        return np.nan, np.nan
    losses = -ret
    var = float(np.quantile(losses, eta))
    tail = losses[losses >= var]
    cvar = float(tail.mean()) if len(tail) else var
    return var, cvar


def performance_metrics(ret, periods_per_year=12):
    ret = pd.Series(ret).dropna()
    if ret.empty:
        return {
            "n_months": 0,
            "ann_return": np.nan,
            "ann_vol": np.nan,
            "sharpe": np.nan,
            "max_drawdown": np.nan,
            "VaR_95_loss": np.nan,
            "CVaR_95_loss": np.nan,
            "final_wealth": np.nan,
        }
    wealth = (1.0 + ret).cumprod()
    ann_return = float(wealth.iloc[-1] ** (periods_per_year / len(ret)) - 1.0)
    ann_vol = float(ret.std(ddof=1) * np.sqrt(periods_per_year))
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan
    var95, cvar95 = empirical_var_cvar(ret, 0.95)
    return {
        "n_months": int(len(ret)),
        "ann_return": ann_return,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "max_drawdown": max_drawdown(ret),
        "VaR_95_loss": var95,
        "CVaR_95_loss": cvar95,
        "final_wealth": float(wealth.iloc[-1]),
    }


metric_rows = []
turnover_rows = []

for (gamma, regime_source), g in strategy_returns.groupby(["gamma", "regime_source"]):
    pivot = g.pivot(index="month", columns="strategy", values="return").sort_index()
    pivot["Equal_Weight_ETF"] = returns.loc[pivot.index, asset_cols].mean(axis=1)
    if "SPY" in returns.columns:
        pivot["SPY_Benchmark"] = returns.loc[pivot.index, "SPY"]

    for strategy in pivot.columns:
        row = {"gamma": gamma, "regime_source": regime_source, "strategy": strategy}
        row.update(performance_metrics(pivot[strategy]))
        metric_rows.append(row)

    wg = strategy_weights[
        (strategy_weights["gamma"] == gamma)
        & (strategy_weights["regime_source"] == regime_source)
    ]
    for strategy, sg in wg.groupby("strategy"):
        w = sg.sort_values("rebalance_month")[asset_cols]
        turnover = w.diff().abs().sum(axis=1).dropna()
        turnover_rows.append({
            "gamma": gamma,
            "regime_source": regime_source,
            "strategy": strategy,
            "avg_turnover": float(turnover.mean()) if len(turnover) else np.nan,
            "max_turnover": float(turnover.max()) if len(turnover) else np.nan,
        })

metrics = pd.DataFrame(metric_rows)
turnover = pd.DataFrame(turnover_rows)
metrics = metrics.merge(turnover, on=["gamma", "regime_source", "strategy"], how="left")

strategy_returns.to_csv(OUT_DIR / "strategy_monthly_returns.csv", index=False)
strategy_weights.to_csv(OUT_DIR / "strategy_weights.csv", index=False)
diagnostics.to_csv(OUT_DIR / "optimizer_diagnostics.csv", index=False)
metrics.to_csv(OUT_DIR / "performance_metrics.csv", index=False)

display(metrics.sort_values(["gamma", "regime_source", "strategy"]))
print("Saved outputs to:", OUT_DIR)


## 5. Optimizer Diagnostics, Robust Penalty Terms, and Plots

The next code cell summarizes why the optimizers behave differently. It saves:

- `weight_concentration_diagnostics.csv`
- `top_strategies_by_sharpe.csv`
- cumulative wealth plots by \(\gamma\) and regime source

### Maximum portfolio weight

For each optimized portfolio:

$$
\max_j x_{t,j}
$$

is reported as `max_weight`.

**Intuition.** This shows whether the optimizer is concentrating in one ETF or spreading capital across several ETFs.

### Effective number of assets

The notebook reports:

$$
N_{\mathrm{eff}}
=
\frac{1}{\sum_{j=1}^{I}x_{t,j}^{2}}.
$$

**Intuition.** If the portfolio is equally weighted across all 10 ETFs, then \(N_{\mathrm{eff}}\approx 10\). If the portfolio is concentrated in only a few ETFs, \(N_{\mathrm{eff}}\) is much smaller.

### Robust norm value

With the default `l1` Wasserstein ground norm, the dual norm used in the penalty is:

$$
\|x_t\|_\infty
=
\max_j x_{t,j}.
$$

In the linear program, this is represented by an auxiliary variable \(z_t\):

$$
x_{t,j}\leq z_t
\quad
\forall j,
\qquad
z_t = \|x_t\|_\infty.
$$

### Robust penalty component

For DR-CVaR and RSDR-CVaR, the reported robust penalty component is:

$$
\mathrm{Penalty}_t
=
\frac{\theta_t}{1-\eta}z_t.
$$

For SAA-CVaR and RS-CVaR:

$$
\theta_t=0,
$$

so the robust penalty is zero.

For DR-CVaR:

$$
\theta_t = \gamma N^{-1/I}.
$$

For RSDR-CVaR:

$$
\theta_t = \theta_{\mathrm{RSDR},t}
=
\sum_{k=1}^{K}w_{t,k}\gamma N_k^{-1/I}.
$$

**Intuition.** The penalty is the extra cost of not fully trusting the empirical distribution. If \(\theta_t\) is larger, the optimizer becomes more conservative. Under the `l1` setup, a concentrated portfolio has a larger \(\|x_t\|_\infty\), so the optimizer may prefer more diversified weights.

### Regime confidence diagnostics

The notebook also reports:

$$
\max_k w_{t,k},
$$

as `regime_weight_max`, and the previous month's largest state probability as `predicted_state_prob`.

**Intuition.** These diagnostics show whether RS-CVaR and RSDR-CVaR are receiving a clear regime signal or a diffuse one. A clear regime signal can make the regime-weighted strategies meaningfully different from SAA-CVaR and DR-CVaR.


In [ ]:

summary = diagnostics.groupby(["gamma", "regime_source", "strategy"])[
    ["max_weight", "effective_n_assets", "theta_used", "robust_penalty_component", "predicted_state_prob", "regime_weight_max"]
].mean().reset_index()

summary.to_csv(OUT_DIR / "weight_concentration_diagnostics.csv", index=False)
display(summary)

top_by_sharpe = metrics.sort_values("sharpe", ascending=False).head(20)
top_by_sharpe.to_csv(OUT_DIR / "top_strategies_by_sharpe.csv", index=False)
display(top_by_sharpe)

for gamma in GAMMA_GRID:
    for regime_source in strategy_returns["regime_source"].unique():
        g = strategy_returns[
            (strategy_returns["gamma"] == gamma)
            & (strategy_returns["regime_source"] == regime_source)
        ]
        pivot = g.pivot(index="month", columns="strategy", values="return").sort_index()
        pivot["Equal_Weight_ETF"] = returns.loc[pivot.index, asset_cols].mean(axis=1)
        if "SPY" in returns.columns:
            pivot["SPY_Benchmark"] = returns.loc[pivot.index, "SPY"]

        wealth = (1.0 + pivot).cumprod()
        plt.figure(figsize=(11, 5))
        for col in wealth.columns:
            lw = 2.5 if "RSDR" in col else 1.4
            plt.plot(wealth.index, wealth[col], label=col, linewidth=lw)
        plt.title(f"Cumulative Wealth: {regime_source}, gamma={gamma}")
        plt.ylabel("Growth of $1")
        plt.xlabel("Month")
        plt.grid(alpha=0.25)
        plt.legend()
        plt.tight_layout()
        fig_path = OUT_DIR / f"cumulative_wealth_gamma_{gamma_label(gamma)}_{regime_source}.png"
        plt.savefig(fig_path, dpi=160, bbox_inches="tight")
        plt.close()

print("Saved cumulative wealth plots to:", OUT_DIR)
